← [Overview](00_overview.ipynb)

# Agglomerative clustering: hierarchical and contiguous

Both methods in this notebook use **Ward agglomerative clustering** — bottom-up
merging that minimises the increase in within-cluster variance at each step. They
differ only in whether merges are restricted to temporally adjacent periods.

> **Relationship to segmentation:** `contiguous` (period-level) and `SegmentConfig`
> (timestep-level within a period) are the **same algorithm** applied at different
> granularities. See [Segmentation](05_segmentation.ipynb) for the timestep-level view.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.figure_factory as ff
import plotly.io as pio
from sklearn.cluster import AgglomerativeClustering

import tsam
from tsam import ClusterConfig

pio.renderers.default = "notebook_connected"

# --------------------------------------------------------------------------
# Load the shared tiny dataset produced by 01_preprocessing (../tiny.csv) and
# rebuild the normalized period matrix D.
# --------------------------------------------------------------------------
tiny = pd.read_csv("../tiny.csv", index_col=0, parse_dates=True)

col_min = tiny.min()
col_max = tiny.max()
normalized = (tiny - col_min) / (col_max - col_min)
N_TIMESTEPS, N_PERIODS, N_ATTRS = 4, 6, 2
D_arr = normalized.values.reshape(N_PERIODS, N_TIMESTEPS * N_ATTRS)

# Real dataset
raw = pd.read_csv("../testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]
UNITS = {"GHI": "W/m²", "T": "°C", "Wind": "m/s", "Load": "MW"}
print("tiny:", tiny.shape, "  real:", data.shape)

tiny: (24, 2)   real: (1008, 4)


---

## 1  Hierarchical clustering (unconstrained Ward)

**Mechanism:** bottom-up agglomeration using **Ward linkage**:

1. Start with every period in its own cluster.
2. Find the pair of clusters whose merge **increases total within-cluster
   variance the least** (Ward criterion).
3. Merge them.
4. Repeat until $k$ clusters remain.

The **Ward merge cost** between clusters $A$ and $B$ is:

$$
\Delta(A, B) = \frac{|A| \cdot |B|}{|A| + |B|} \| \bar{x}_A - \bar{x}_B \|^2
$$

where $|A|$, $|B|$ are cluster sizes and $\bar{x}_A$, $\bar{x}_B$ are their centroids.

The hierarchy is captured in a **dendrogram**. Any number of clusters can be read
off by cutting at different heights.

**TSAM configuration for hierarchical clustering:**

In [2]:
from tsam import ClusterConfig
import tsam

# Hierarchical: bottom-up Ward agglomerative clustering, unconstrained.
# representation defaults to 'medoid' for hierarchical.
cfg_hierarchical = ClusterConfig(method="hierarchical", representation="medoid")
print(cfg_hierarchical)

# Full aggregate call:
# result = tsam.aggregate(
#     df,
#     n_clusters=k,
#     period_duration="1D",
#     cluster=cfg_hierarchical,
# )

ClusterConfig(include_period_sums=False, method='hierarchical', representation='medoid', scale_by_column_means=False, solver='highs', use_duration_curves=False)


In [3]:
import scipy.cluster.hierarchy as sch

labels = [f"day_{i}" for i in range(N_PERIODS)]

fig_dendro = ff.create_dendrogram(
    D_arr,
    labels=labels,
    linkagefun=lambda x: sch.ward(x),
    colorscale=px.colors.qualitative.Set1[:6],
)
fig_dendro.update_layout(
    title=(
        "Ward dendrogram — 6 tiny periods<br>"
        "<sup>Cutting at k=3 gives three groups; sunny days (0,1) merge first</sup>"
    ),
    yaxis_title="Ward distance",
    xaxis_title="Period",
)
fig_dendro.show()

In [4]:
result_hier_tiny = tsam.aggregate(
    tiny,
    n_clusters=3,
    period_duration="1D",
    cluster=ClusterConfig(method="hierarchical"),
)
print("Hierarchical on tiny — assignments:", result_hier_tiny.cluster_assignments)
print("Cluster counts:", result_hier_tiny.cluster_counts)

Hierarchical on tiny — assignments: [1 1 2 2 0 0]
Cluster counts: {0: 2.0, 1: 2.0, 2: 2.0}


In [5]:
result_hier = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    cluster=ClusterConfig(method="hierarchical"),
)
print("Hierarchical (real) — weighted RMSE:", round(result_hier.accuracy.weighted_rmse, 4))
result_hier.plot.cluster_members(columns=["Load"], title="Hierarchical: Load cluster members")

Hierarchical (real) — weighted RMSE: 0.1235


### Where the clusters fall on the calendar

Unconstrained Ward groups days purely by shape, so a single cluster can gather days from anywhere in the six weeks — the colours are interleaved across the calendar.

In [6]:
result_hier.plot.clusters_over_time(
    columns=["Load"],
    units=UNITS,
    title="Hierarchical (unconstrained): clusters scattered across the calendar",
)

---

## 2  Contiguous clustering — Ward with a temporal-adjacency constraint

> **Common misconception corrected:** `contiguous` is sometimes described as
> "time-based". This is wrong. Algorithmically it is **Ward agglomerative
> clustering with an adjacency-matrix connectivity constraint** — identical to
> `hierarchical`, except the connectivity matrix restricts merges to
> **immediately adjacent periods** only. It is therefore:
>
> * **Feature-based** (uses Ward / within-cluster variance, i.e. feature similarity)
> * **With a time-contiguity constraint** (can only merge neighbours)
>
> It sits in the **feature-based** row of the Hoffmann taxonomy.

**Connection to segmentation:** `contiguous` and
[segmentation](05_segmentation.ipynb) are the **same algorithm**
(Ward + adjacency constraint) applied at different granularities:
* `contiguous` merges **periods** (rows of the D matrix)
* segmentation merges **timesteps** within each period

Internally tsam builds the bidiagonal adjacency matrix $I(|i-j|=1)$ and
passes it as `connectivity` to scikit-learn's `AgglomerativeClustering`.

**TSAM configuration for contiguous clustering:**

In [7]:
# Contiguous: Ward agglomerative clustering with temporal-adjacency constraint.
# Feature-based (Ward variance criterion), not time-based.
# representation defaults to 'medoid' for contiguous.
cfg_contiguous = ClusterConfig(method="contiguous", representation="medoid")
print(cfg_contiguous)

# Full aggregate call:
# result = tsam.aggregate(
#     df,
#     n_clusters=k,
#     period_duration="1D",
#     cluster=cfg_contiguous,
# )

ClusterConfig(include_period_sums=False, method='contiguous', representation='medoid', scale_by_column_means=False, solver='highs', use_duration_curves=False)


In [8]:
# Illustrate the adjacency constraint
adj = np.eye(N_PERIODS, k=1) + np.eye(N_PERIODS, k=-1)  # bidiagonal
print("Adjacency matrix (only direct neighbours may merge):")
print(adj.astype(int))

contiguous_sk = AgglomerativeClustering(
    n_clusters=3, linkage="ward", connectivity=adj
)
labels_contiguous = contiguous_sk.fit_predict(D_arr)
print("\nContiguous assignments (scikit-learn):")
for i, lbl in enumerate(labels_contiguous):
    print(f"  day_{i} -> cluster {lbl}")

Adjacency matrix (only direct neighbours may merge):
[[0 1 0 0 0 0]
 [1 0 1 0 0 0]
 [0 1 0 1 0 0]
 [0 0 1 0 1 0]
 [0 0 0 1 0 1]
 [0 0 0 0 1 0]]

Contiguous assignments (scikit-learn):
  day_0 -> cluster 1
  day_1 -> cluster 1
  day_2 -> cluster 2
  day_3 -> cluster 2
  day_4 -> cluster 0
  day_5 -> cluster 0


In [9]:
result_cont_tiny = tsam.aggregate(
    tiny,
    n_clusters=3,
    period_duration="1D",
    cluster=ClusterConfig(method="contiguous"),
)
print("Contiguous (tiny) — assignments:", result_cont_tiny.cluster_assignments)
print("Cluster counts:", result_cont_tiny.cluster_counts)

result_cont = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    cluster=ClusterConfig(method="contiguous"),
)
print("\nContiguous (real) — weighted RMSE:", round(result_cont.accuracy.weighted_rmse, 4))

Contiguous (tiny) — assignments: [1 1 2 2 0 0]
Cluster counts: {0: 2.0, 1: 2.0, 2: 2.0}



Contiguous (real) — weighted RMSE: 0.1445


Contiguous Ward may only merge **adjacent** periods, so every cluster is a single unbroken stretch of the calendar — the same six weeks now split into consecutive date blocks. This is the only difference from the plot above.

In [10]:
result_cont.plot.clusters_over_time(
    columns=["Load"],
    units=UNITS,
    title="Contiguous: each cluster is a consecutive date block",
)

In [11]:
# Accuracy comparison
summary = pd.DataFrame({
    "method": ["hierarchical", "contiguous"],
    "weighted_rmse": [
        round(result_hier.accuracy.weighted_rmse, 4),
        round(result_cont.accuracy.weighted_rmse, 4),
    ],
    "note": [
        "any merge order",
        "adjacent merges only",
    ],
})
print("Agglomerative clustering — accuracy on 6-week real dataset (k=6):")
summary

Agglomerative clustering — accuracy on 6-week real dataset (k=6):


,method,weighted_rmse,note
0,hierarchical,0.1235,any merge order
1,contiguous,0.1445,adjacent merges only


---

**Up next:**
* [Averaging](04_averaging.ipynb) — time-based block grouping
* [Segmentation](05_segmentation.ipynb) — the timestep-level analogue of `contiguous`